# Visual Asset Auditing System — v2

Brand logos · UI components · image→image · text→image · person search · exact reverse-image.

### What changed and why

| # | Change | Why |
|---|---|---|
| 1 | Vector arm rewritten to `ORDER BY embedding <=> $q LIMIT k` on a **materialized unique-image table** | v1 computed the distance as a CTE column with no limit, then `DISTINCT ON (content_hash)` over the whole set. Both force a full scan + full sort of all 99L rows. **HNSW was never used.** Query-time dedup and ANN indexing are mutually exclusive. |
| 2 | Contrastive negative **folded into the query vector** | `(embedding<=>$1) - w*(embedding<=>$2)` is arithmetic on two distances — no vector index can ever serve it. Folding keeps lookalike suppression *and* stays indexable. |
| 3 | **Dual query vectors** (image-only + text-only), fused by RRF | `gemini-embedding-2` returns ONE **aggregated** embedding when a request carries several inputs. v1 passed `[image, 1000 chars of prose]`, blending them and dragging the query off the image manifold. Separate `Content` objects keep "this logo" and "the OLD version" as independent signals. |
| 4 | Reranker made **bounded + additive** | v1 multiplied RRF by up to ~48× (`2³` tags × `3` keyword × `2` vector). A true #1 vector match scored ~0.011 while a rank-#3000 item with generic tags scored ~0.26 — **24× higher**. Junk outranked real matches. |
| 5 | Keyword density **deleted** | It was `desc.count(kw)` — substring matching, so "art" hits "part"/"start"/"artificial". Also redundant with the FTS arm. |
| 6 | FTS switched from `plainto_tsquery` (**AND**) to OR-joined `to_tsquery` | v1 required *every* keyword to be present — severe recall loss. |
| 7 | Tags resolved by **`pg_trgm` against the corpus**; no vocabulary in the prompt | Fixes 70k tags → 429. Also, top-300-**by-frequency** returns `Text, Font, Rectangle` — the *least* discriminative tags, which is why the tag arm never helped. |
| 8 | Tags demoted from an RRF arm to a **bounded boost + optional filter** | An arm with no ORDER BY was feeding RRF a random permutation. |
| 9 | Person search backed by an **`asset_faces` per-face embedding table** | A 30px face contributes ~nothing to a whole-image vector. This is a retrieval blind spot — no prompt or threshold can recover it. |
| 10 | `EXACT_MATCH` served by **`content_hash`**, not the LLM | O(1), exact precision, zero cost. |
| 11 | Stage 1's two Gemini-Pro calls **merged into one** | Halves Stage-1 latency and cost. |
| 12 | Reference image + protocol **context-cached** | IMAGE 0 is byte-identical across every candidate call in a run. |
| 13 | Kneedle + 3 unbounded rescues → **hard candidate budget** | Rescue passes could dump *every* face-tagged row into Edge, making cost and p95 unpredictable. |

**Run order:** Config → Connect → Preflight → *(one-time)* DDL → Stage 1 → Stage 2.

In [ ]:
# !pip install -q google-genai google-cloud-alloydb-connector[asyncpg] "sqlalchemy[asyncio]" \
#     asyncpg pgvector pandas numpy pillow google-cloud-storage google-cloud-vision nest_asyncio

try:
    from google.colab import auth; auth.authenticate_user()
except Exception:
    pass

In [ ]:
# ============================== CONFIG ==============================
import math, io, os, re, json, time, asyncio, random
from typing import Literal, Optional, List, Dict, Tuple, Any
import numpy as np, pandas as pd
import google.genai as genai
from google.genai import types

PROJECT_ID, REGION = "YOUR_PROJECT_ID", "YOUR_REGION"
ALLOYDB_CLUSTER, ALLOYDB_INSTANCE = "YOUR_CLUSTER_ID", "YOUR_INSTANCE_ID"
DB_USER, DB_PASSWORD, DB_NAME, DB_SCHEMA = "YOUR_DB_USER", "YOUR_DB_PASSWORD", "YOUR_DB_NAME", "public"

ORCHESTRATOR_MODEL = "gemini-2.5-pro"     # Stage 1: forensics + audit config (reasoning)
INFERENCE_MODEL    = "gemini-2.5-flash"   # Stage 3: per-candidate audit (speed + cost)

# MUST be identical to the model+dim that wrote visual_assets.embedding.
# gemini-embedding-2 is natively multimodal: text, images, video, audio and PDFs share ONE
# space, so text->image and image->image both run off this single column.
EMBEDDING_MODEL, EMBEDDING_DIM = "gemini-embedding-2", 768

BASE_TABLE   = "visual_assets"          # ~99L rows: one row per (image, page) occurrence
UNIQUE_TABLE = "visual_assets_unique"   # ~6L rows: one row per unique image (content_hash)
FACES_TABLE  = "asset_faces"
VOCAB_VIEW   = "tag_vocab"
USE_UNIQUE_TABLE = True

def TBL() -> str:
    return f"{DB_SCHEMA}.{UNIQUE_TABLE}" if USE_UNIQUE_TABLE else f"{DB_SCHEMA}.{BASE_TABLE}"

client = genai.Client(project=PROJECT_ID, location=REGION, vertexai=True)

ARM_LIMIT = 400          # RRF contribution past ~rank 300 is negligible: 1/(60+300)
RRF_K     = 60

EF_SEARCH = {"LOGO_SIMILARITY":120, "UI_COMPONENT":120, "PERSON_SEARCH":200,
             "EXACT_MATCH":64, "GENERAL_ASSET":120}

# Arms: vec_image (image-only vector) · vec_text (intent-only vector) · fts · face.
# Renormalized at runtime over whichever arms actually produced results.
RRF_WEIGHTS = {
    "LOGO_SIMILARITY": {"vec_image":0.55, "vec_text":0.25, "fts":0.20, "face":0.00},
    "UI_COMPONENT":    {"vec_image":0.45, "vec_text":0.35, "fts":0.20, "face":0.00},
    "PERSON_SEARCH":   {"vec_image":0.30, "vec_text":0.05, "fts":0.05, "face":0.60},
    "EXACT_MATCH":     {"vec_image":1.00, "vec_text":0.00, "fts":0.00, "face":0.00},
    "GENERAL_ASSET":   {"vec_image":0.40, "vec_text":0.40, "fts":0.20, "face":0.00},
}
NEGATIVE_WEIGHT = {"LOGO_SIMILARITY":0.30, "UI_COMPONENT":0.20, "PERSON_SEARCH":0.00,
                   "EXACT_MATCH":0.00, "GENERAL_ASSET":0.15}
SAFEGUARD_DIST  = {"LOGO_SIMILARITY":0.28, "UI_COMPONENT":0.32, "PERSON_SEARCH":0.50,
                   "EXACT_MATCH":0.10, "GENERAL_ASSET":0.35}
# Hard cap on candidates reaching the LLM — your cost and p95 dial.
BUDGET          = {"LOGO_SIMILARITY":40, "UI_COMPONENT":40, "PERSON_SEARCH":80,
                   "EXACT_MATCH":15, "GENERAL_ASSET":40}
CONF_THRESHOLD  = {"LOGO_SIMILARITY":72, "UI_COMPONENT":70, "PERSON_SEARCH":50,
                   "EXACT_MATCH":88, "GENERAL_ASSET":68}
# A category-wide sweep must not be judged as strictly as "find THIS exact asset".
BREADTH_CONF_ADJ = {"EXACT_INSTANCE": +10, "SAME_BRAND_ANY_VARIANT": 0, "SAME_CATEGORY": -12}

MAX_WORKERS, USE_CONTEXT_CACHE, CACHE_TTL = 20, True, "900s"
FACE_DIM = 768
PERSON_FACE_TAGS = ["Person","People","Face","Head","Human","Portrait","Selfie","Smile",
                    "Facial expression","Adult","Man","Woman","Child","Forehead","Cheek",
                    "Chin","Hair","Beard","Hairstyle","Profile picture"]

print(f"Embedding : {EMBEDDING_MODEL} @ {EMBEDDING_DIM}d   Search table: {TBL()}")

In [ ]:
# ============================== CONNECTION ==============================
import asyncpg, nest_asyncio
from sqlalchemy.ext.asyncio import create_async_engine
from google.cloud.alloydb.connector import IPTypes, AsyncConnector
from pgvector.asyncpg import register_vector
from contextlib import asynccontextmanager
nest_asyncio.apply()

_cache = {}

async def get_engine():
    if 'e' in _cache:
        return _cache['e']
    connector = AsyncConnector(refresh_strategy="lazy")

    async def getconn():
        uri = ALLOYDB_INSTANCE
        if not uri.startswith("projects/"):
            uri = (f"projects/{PROJECT_ID}/locations/{REGION}"
                   f"/clusters/{ALLOYDB_CLUSTER}/instances/{ALLOYDB_INSTANCE}")
        return await asyncio.wait_for(
            connector.connect(uri, "asyncpg", user=DB_USER, password=DB_PASSWORD,
                              db=DB_NAME, enable_iam_auth=False, ip_type=IPTypes.PUBLIC),
            timeout=10.0)

    _cache['e'] = create_async_engine("postgresql+asyncpg://", async_creator=getconn,
                                      pool_size=12, max_overflow=24, pool_pre_ping=True)
    _cache['c'] = connector
    return _cache['e']


@asynccontextmanager
async def db_conn(vector: bool = False):
    """Yields a raw asyncpg connection. `vector=True` registers the pgvector codec."""
    engine = await get_engine()
    conn = await engine.connect()
    try:
        raw = await conn.get_raw_connection()
        db = raw.driver_connection
        if vector:
            await register_vector(db)
        yield db
    finally:
        await conn.close()

## Preflight

Confirms the three things everything else depends on: the stored vector dimension matches the
query model, `content_hash` is actually populated (v1's `COALESCE(content_hash, gcs_raw_path)`
implies it may not be — in which case you were never deduping to 6L), and an HNSW index exists
**built with `vector_cosine_ops`** (an `l2_ops` index cannot serve `<=>`, so even a correct
query silently falls back to a full scan).

`self_check` then embeds real rows through the *query* path and verifies they retrieve
themselves at distance ≈ 0 — proving query and corpus share a vector space, and revealing
whether the corpus was embedded image-only or image+text.

In [ ]:
async def preflight(deep: bool = True) -> dict:
    rep = {}
    async with db_conn() as db:
        cols = await db.fetch("SELECT column_name, udt_name FROM information_schema.columns "
                              "WHERE table_schema=$1 AND table_name=$2 ORDER BY ordinal_position",
                              DB_SCHEMA, BASE_TABLE)
        rep["columns"] = {r["column_name"]: r["udt_name"] for r in cols}
        print(f"── {BASE_TABLE} columns ──")
        for k, v in rep["columns"].items():
            print(f"   {k:<26} {v}")

        dim = await db.fetchval(f"SELECT vector_dims(embedding) FROM {DB_SCHEMA}.{BASE_TABLE} "
                                f"WHERE embedding IS NOT NULL LIMIT 1")
        rep["dim"] = dim
        flag = "OK" if dim == EMBEDDING_DIM else f"MISMATCH — config says {EMBEDDING_DIM}"
        print(f"\n── stored vector dim: {dim}  [{flag}]")

        idx = await db.fetch("SELECT indexname, indexdef FROM pg_indexes "
                             "WHERE schemaname=$1 AND tablename=ANY($2)",
                             DB_SCHEMA, [BASE_TABLE, UNIQUE_TABLE, FACES_TABLE])
        rep["cosine_hnsw"] = any("hnsw" in r["indexdef"].lower()
                                 and "vector_cosine_ops" in r["indexdef"].lower() for r in idx)
        print("\n── indexes ──")
        for r in idx:
            print(f"   {r['indexname']}: {r['indexdef']}")
        if not rep["cosine_hnsw"]:
            print("\n   NO hnsw(vector_cosine_ops) index → every `<=>` query is a FULL SCAN.")
            print("   This is the single biggest source of search latency. Run build_unique_table().")

        for t in (UNIQUE_TABLE, FACES_TABLE, VOCAB_VIEW):
            ok = await db.fetchval("SELECT to_regclass($1) IS NOT NULL", f"{DB_SCHEMA}.{t}")
            rep[t] = bool(ok)
            print(f"── {t:<24} {'present' if ok else 'MISSING'}")

        if deep:
            s = await db.fetchrow(f"SELECT count(*) total, count(content_hash) hashed, "
                                  f"count(DISTINCT content_hash) uniq FROM {DB_SCHEMA}.{BASE_TABLE}")
            rep["counts"] = dict(s)
            print(f"\n── rows: {s['total']:,} total | {s['hashed']:,} hashed "
                  f"({100*s['hashed']/max(s['total'],1):.1f}%) | {s['uniq']:,} unique")
            if s["hashed"] < s["total"]:
                print(f"   {s['total']-s['hashed']:,} rows have NULL content_hash — they cannot be "
                      f"deduped and will be audited redundantly. Populate it first.")
    globals()["PREFLIGHT"] = rep
    return rep


def _embed(contents: list, dim: int = EMBEDDING_DIM) -> List[float]:
    """One embedding call. NOTE: several inputs in ONE call => ONE aggregated vector."""
    return client.models.embed_content(
        model=EMBEDDING_MODEL, contents=contents,
        config=types.EmbedContentConfig(output_dimensionality=dim)).embeddings[0].values


def _bytes(path: str) -> bytes:
    from google.cloud import storage
    if path.startswith("gs://"):
        b, o = path.replace("gs://", "").split("/", 1)
        return storage.Client(project=PROJECT_ID).bucket(b).blob(o).download_as_bytes()
    with open(path, "rb") as f:
        return f.read()


async def self_check(n: int = 5):
    """Do query-path embeddings land in the same space as the corpus? And how was it ingested?"""
    async with db_conn(vector=True) as db:
        rows = await db.fetch(f"SELECT asset_id, gcs_raw_path, gemini_description FROM {TBL()} "
                              f"WHERE embedding IS NOT NULL ORDER BY random() LIMIT {n}")
        modes = {"image_only": [], "text_only": [], "image_plus_text": []}
        for r in rows:
            try:
                img = _bytes(r["gcs_raw_path"])
                desc = str(r["gemini_description"] or "")[:2000]
                ip = types.Part.from_bytes(data=img, mime_type="image/png")
                trials = {"image_only": [ip], "text_only": [desc], "image_plus_text": [ip, desc]}
                for name, contents in trials.items():
                    if name == "text_only" and not desc:
                        continue
                    q = _embed(contents)
                    top = await db.fetchrow(
                        f"SELECT asset_id, embedding <=> $1::vector d FROM {TBL()} "
                        f"ORDER BY embedding <=> $1::vector LIMIT 1", q)
                    modes[name].append((top["asset_id"] == r["asset_id"], float(top["d"])))
            except Exception as e:
                print(f"  asset {r['asset_id']}: {e}")

    print("\n  composition        self-hit   mean top-1 distance")
    best, best_d = None, 9.9
    for name, res in modes.items():
        if not res:
            continue
        rate = sum(int(h) for h, _ in res) / len(res)
        md = float(np.mean([d for _, d in res]))
        print(f"  {name:<18} {rate:>6.0%}     {md:.4f}")
        if md < best_d:
            best, best_d = name, md
    if best_d < 0.15:
        print(f"\n  PASS — same vector space. Corpus looks ingested as: {best}")
    else:
        print("\n  FAIL — query path is NOT in the corpus vector space. "
              "Every vector result so far has been effectively random. Fix EMBEDDING_MODEL first.")
    return modes

# try: _ = asyncio.run(preflight())
# try: _ = asyncio.run(self_check())

## One-time DDL

Materializing the 6L unique images is what makes the ANN index usable — and it cuts LLM
inference cost ~16× by never auditing the same image twice.

In [ ]:
async def build_unique_table():
    async with db_conn() as db:
        dim = (globals().get("PREFLIGHT", {}) or {}).get("dim") or EMBEDDING_DIM
        print(f"Building {UNIQUE_TABLE} (dim={dim})… several minutes over 99L rows.")
        await db.execute("CREATE EXTENSION IF NOT EXISTS vector")
        await db.execute("CREATE EXTENSION IF NOT EXISTS pg_trgm")
        await db.execute(f"DROP TABLE IF EXISTS {DB_SCHEMA}.{UNIQUE_TABLE} CASCADE")
        await db.execute(f"""
            CREATE TABLE {DB_SCHEMA}.{UNIQUE_TABLE} AS
            SELECT DISTINCT ON (content_hash)
                   asset_id, content_hash, gcs_raw_path, format, vision_tags,
                   gemini_description, asset_filename, page_url, embedding
            FROM {DB_SCHEMA}.{BASE_TABLE}
            WHERE content_hash IS NOT NULL AND embedding IS NOT NULL
            ORDER BY content_hash, asset_id""")
        print(f"  {await db.fetchval(f'SELECT count(*) FROM {DB_SCHEMA}.{UNIQUE_TABLE}'):,} unique images")

        await db.execute(f"ALTER TABLE {DB_SCHEMA}.{UNIQUE_TABLE} ADD PRIMARY KEY (content_hash)")
        # vector_cosine_ops is REQUIRED for `<=>` to be index-served.
        await db.execute(f"""CREATE INDEX ON {DB_SCHEMA}.{UNIQUE_TABLE}
            USING hnsw (embedding vector_cosine_ops) WITH (m=16, ef_construction=64)""")
        await db.execute(f"CREATE INDEX ON {DB_SCHEMA}.{UNIQUE_TABLE} USING gin (vision_tags)")
        await db.execute(f"""CREATE INDEX ON {DB_SCHEMA}.{UNIQUE_TABLE}
            USING gin (to_tsvector('english', coalesce(gemini_description,'')))""")
        await db.execute(f"ANALYZE {DB_SCHEMA}.{UNIQUE_TABLE}")

        # Tag vocabulary — replaces stuffing 70k tags into the prompt.
        await db.execute(f"DROP MATERIALIZED VIEW IF EXISTS {DB_SCHEMA}.{VOCAB_VIEW} CASCADE")
        await db.execute(f"""CREATE MATERIALIZED VIEW {DB_SCHEMA}.{VOCAB_VIEW} AS
            SELECT tag, count(*) AS freq FROM {DB_SCHEMA}.{UNIQUE_TABLE}, unnest(vision_tags) tag
            WHERE tag IS NOT NULL AND length(tag) > 1 GROUP BY tag""")
        await db.execute(f"CREATE INDEX ON {DB_SCHEMA}.{VOCAB_VIEW} USING gin (tag gin_trgm_ops)")
        print(f"  {await db.fetchval(f'SELECT count(*) FROM {DB_SCHEMA}.{VOCAB_VIEW}'):,} tags indexed")
        print("Done.")


async def build_faces_table():
    """Per-face embeddings — the only way small/corner faces become retrievable."""
    async with db_conn() as db:
        await db.execute(f"""CREATE TABLE IF NOT EXISTS {DB_SCHEMA}.{FACES_TABLE} (
            face_id bigserial PRIMARY KEY, content_hash text NOT NULL, asset_id bigint,
            bbox int[], face_area_frac real, detect_conf real,
            face_embedding vector({FACE_DIM}))""")
        await db.execute(f"""CREATE INDEX IF NOT EXISTS {FACES_TABLE}_hnsw ON {DB_SCHEMA}.{FACES_TABLE}
            USING hnsw (face_embedding vector_cosine_ops) WITH (m=16, ef_construction=64)""")
        await db.execute(f"CREATE INDEX IF NOT EXISTS {FACES_TABLE}_ch ON {DB_SCHEMA}.{FACES_TABLE} (content_hash)")
        print(f"{FACES_TABLE} ready — populate with backfill_faces().")


def detect_faces(img_bytes: bytes) -> List[dict]:
    """Cloud Vision face boxes. Returns [{bbox, conf, area_frac}]."""
    from google.cloud import vision
    from PIL import Image
    resp = vision.ImageAnnotatorClient().face_detection(image=vision.Image(content=img_bytes))
    W, H = Image.open(io.BytesIO(img_bytes)).size
    out = []
    for f in resp.face_annotations:
        v = f.bounding_poly.vertices
        x0, y0 = max(min(p.x for p in v), 0), max(min(p.y for p in v), 0)
        x1, y1 = min(max(p.x for p in v), W), min(max(p.y for p in v), H)
        if x1 > x0 and y1 > y0:
            out.append({"bbox": [x0, y0, x1, y1], "conf": float(f.detection_confidence),
                        "area_frac": ((x1-x0)*(y1-y0)) / float(W*H)})
    return out


def crop_face(img_bytes: bytes, bbox: List[int], pad: float = 0.25) -> bytes:
    """Crop with margin — context around the face improves embedding quality."""
    from PIL import Image
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    x0, y0, x1, y1 = bbox
    px, py = int((x1-x0)*pad), int((y1-y0)*pad)
    crop = img.crop((max(x0-px, 0), max(y0-py, 0), min(x1+px, img.width), min(y1+py, img.height)))
    if min(crop.size) < 112:                       # upscale tiny faces before embedding
        s = 112 / min(crop.size)
        crop = crop.resize((int(crop.width*s), int(crop.height*s)), Image.LANCZOS)
    buf = io.BytesIO(); crop.save(buf, format="JPEG", quality=95)
    return buf.getvalue()


async def backfill_faces(limit: Optional[int] = None, batch: int = 200):
    """One-time pass over unique images: detect faces → crop → embed → store."""
    from concurrent.futures import ThreadPoolExecutor
    async with db_conn(vector=True) as db:
        done = await db.fetchval(f"SELECT count(DISTINCT content_hash) FROM {DB_SCHEMA}.{FACES_TABLE}")
        rows = await db.fetch(f"""
            SELECT u.content_hash, u.asset_id, u.gcs_raw_path FROM {DB_SCHEMA}.{UNIQUE_TABLE} u
            WHERE NOT EXISTS (SELECT 1 FROM {DB_SCHEMA}.{FACES_TABLE} f
                              WHERE f.content_hash = u.content_hash)
            {f'LIMIT {limit}' if limit else ''}""")
        print(f"{done:,} already processed · {len(rows):,} remaining")

        def work(r):
            try:
                b = _bytes(r["gcs_raw_path"])
                return [(r["content_hash"], r["asset_id"], f["bbox"], f["area_frac"], f["conf"],
                         _embed([types.Part.from_bytes(data=crop_face(b, f["bbox"]),
                                                       mime_type="image/jpeg")], FACE_DIM))
                        for f in detect_faces(b)]
            except Exception:
                return []

        with ThreadPoolExecutor(max_workers=16) as ex:
            for i in range(0, len(rows), batch):
                recs = [x for sub in ex.map(work, rows[i:i+batch]) for x in sub]
                if recs:
                    await db.executemany(
                        f"INSERT INTO {DB_SCHEMA}.{FACES_TABLE} "
                        f"(content_hash, asset_id, bbox, face_area_frac, detect_conf, face_embedding) "
                        f"VALUES ($1,$2,$3,$4,$5,$6)", recs)
                print(f"  {min(i+batch, len(rows)):,}/{len(rows):,} · +{len(recs)} faces")
        await db.execute(f"ANALYZE {DB_SCHEMA}.{FACES_TABLE}")

# asyncio.run(build_unique_table()); asyncio.run(build_faces_table()); asyncio.run(backfill_faces())

## Stage 1 — Audit config (one Pro call)

v1 spent two sequential Gemini-Pro calls here: a forensic pass, then a config pass. Merged into
one multimodal call.

Two prompt changes matter most:

- **`tag_guesses` are free text**, resolved against the corpus by `pg_trgm` afterwards. Nothing
  from the tag vocabulary enters the prompt, so the 70k-tag → 429 cannot happen.
- v1 instructed *"Do NOT name brands from memory."* That is removed. For **"find all the old
  logos"**, knowing *"this is the legacy mark, superseded by the 2020 flat redesign"* is the
  entire discriminator. Visual criteria still come only from what is visible.

In [ ]:
from pydantic import BaseModel, Field, create_model

SEARCH_MODES = Literal["LOGO_SIMILARITY","UI_COMPONENT","PERSON_SEARCH","EXACT_MATCH","GENERAL_ASSET"]

class AuditConfig(BaseModel):
    search_mode: SEARCH_MODES = Field(description=(
        "LOGO_SIMILARITY: brand logos/icons/identity marks. UI_COMPONENT: buttons, modals, banners, "
        "cookie notices, payment flows. PERSON_SEARCH: a specific person, even a tiny thumbnail. "
        "EXACT_MATCH: near-identical duplicate (reverse image search). GENERAL_ASSET: anything else."))
    intent_breadth: Literal["EXACT_INSTANCE","SAME_BRAND_ANY_VARIANT","SAME_CATEGORY"] = Field(
        description=(
            "How wide the user's net is — independent of search_mode, and inferred from how they "
            "phrased the goal.\n"
            "EXACT_INSTANCE: this specific asset only ('where is THIS image used').\n"
            "SAME_BRAND_ANY_VARIANT: this brand/subject in any variant, era, colourway or lockup "
            "('find the old logos', 'every version of this mark').\n"
            "SAME_CATEGORY: the whole visual class ('upload an Android logo, find all logos', "
            "'all cookie banners'). Here the reference is an EXAMPLE, not the target — "
            "inclusion_criteria must describe the CATEGORY, never this one instance."))
    audit_goal: str = Field(description="Refined, precise restatement of the user's goal.")
    image_description: Optional[str] = Field(None, description=(
        "Forensic description of the reference image: content type; exact shapes, outlines and "
        "proportions; exact visible text with capitalisation and font weight; layout of sub-elements; "
        "design generation (flat/gradient/3D, filled/outlined). For a person, describe ONLY permanent "
        "bone structure — face shape, jawline, eye spacing, nose bridge and tip, brow ridge, hairline. "
        "Never describe clothing, background or lighting for a person."))
    brand_or_identity: Optional[str] = Field(None, description=(
        "If the mark is recognisable, name the brand AND the specific generation/era "
        "(e.g. 'legacy pre-2020 wordmark with gradient'). This is what makes 'find the OLD logo' "
        "work. Leave null if genuinely unrecognisable. Visual criteria must still come only from "
        "what is visible."))
    intent_query: str = Field(description=(
        "One dense sentence describing WHAT TO RETRIEVE, written as if describing the target image "
        "to someone who cannot see it. Embedded on its own as the text search vector — so write "
        "visual nouns, not instructions. Good: 'legacy blue-and-green gradient payment wordmark with "
        "rounded lowercase lettering'. Bad: 'find all the old logos'."))
    inclusion_criteria: List[str] = Field(description="3-6 specific, visually testable properties an image MUST have.")
    exclusion_criteria: List[str] = Field(description="2-4 disqualifiers. Name concrete lookalikes and wrong generations.")
    adjudication_logic: str = Field(description="Numbered STEP 1..N protocol. See the template in the prompt.")
    search_keywords: List[str] = Field(description=(
        "5-8 SINGLE words, OR-matched against Gemini-written descriptions. No phrases."))
    tag_guesses: List[str] = Field(description=(
        "4-8 short free-text guesses at Cloud Vision labels for this target (e.g. 'logo', 'banner', "
        "'cookie consent'). Resolved against the real corpus vocabulary afterwards — do NOT worry "
        "about matching exactly."))
    audit_instructions: str = Field(description="≤120 words of extra guidance. Do not repeat the criteria.")
    extraction_schema: Dict[str, str] = Field(description=(
        "Extra per-image fields as {name: type}. Always include detected_asset_style (string), "
        "is_outdated_or_noncompliant (boolean), is_embedded_in_composite (boolean), "
        "optical_resolution_sufficient (boolean)."))


ADJUDICATION_TEMPLATE = """\
Write adjudication_logic as numbered steps that mirror how a careful human reviewer actually
inspects an image on screen. The inference model always receives IMAGE 0 (reference) and
IMAGE 1 (candidate), so every step must reference them explicitly.

  STEP 1 (LOCATE IN REFERENCE): In IMAGE 0, name the single target element and the exact
    features that identify it: <shape/outline, exact wordmark text, face bone structure>.
  STEP 2 (SCAN CANDIDATE): Sweep IMAGE 1 region by region — top-left → top-right → centre →
    bottom-left → bottom-right. Include elements inside screenshots, hero banners, device
    frames, app screens and thumbnails. Zoom into small and corner regions: the target may be
    30px, cropped, rotated, low-contrast or partially occluded.
  STEP 3 (COMPARE SIDE BY SIDE): Put the located element next to IMAGE 0 and check feature by
    feature: <outline and proportions; exact text spelling, capitalisation and weight;
    arrangement of sub-elements; design generation; for people: jawline, eye spacing, nose,
    brow ridge, hairline>.
  STEP 4 (EXCLUDE): Actively rule out the named lookalikes: <misspelled wordmarks, sibling
    brands, wrong generation, a different person with similar hair or framing>.
  STEP 5 (VERDICT): PASS only if <exact condition>. FAIL if the target is absent, any exclusion
    triggers, or identity cannot be confirmed at the required confidence.

Replace every <...> with specifics for THIS audit. Two reviewers following these steps must
reach the same verdict."""


def _forensic_prompt(goal: str, has_image: bool) -> str:
    return f"""You are a senior visual-asset forensics expert and audit designer.
Produce a complete audit configuration in ONE pass.

USER GOAL: {goal}

{"A reference image is attached. Analyse it before writing anything else." if has_image
 else "No reference image was provided — this is a text-only search. Set image_description to null."}

STEP 1 — CLASSIFY MODE, THEN BREADTH
Pick exactly one search_mode. "Find where this image appears" => EXACT_MATCH.
"Find this person" => PERSON_SEARCH. "Find logos like this / old versions" => LOGO_SIMILARITY.

Then set intent_breadth, which decides how the criteria are written:
  "where is this image used"        => EXACT_INSTANCE
  "find the old/all versions of it" => SAME_BRAND_ANY_VARIANT
  "here is a logo, find all logos"  => SAME_CATEGORY
This distinction matters: under SAME_CATEGORY the attached image is only an EXAMPLE of the
class. Writing criteria that describe that one instance would wrongly reject the whole
category the user actually asked for.

STEP 2 — FORENSIC READ OF THE REFERENCE
Report content type, exact shapes and proportions, exact visible text (spelling, capitalisation,
weight), layout of sub-elements, and design generation. Base visual criteria ONLY on what is
visible. Separately, in brand_or_identity, you MAY name the brand and its generation from
knowledge — that is what makes "find the OLD logo" possible.
For a person: describe ONLY permanent bone structure. Never clothing, hair colour or background.

STEP 3 — CRITERIA
inclusion_criteria: visually testable properties that MUST hold.
exclusion_criteria: concrete disqualifiers — name the actual lookalikes and wrong generations.
COLOUR, GRADIENT AND FINISH — decide by intent, do not blanket-ignore them:
  - If the goal concerns GENERATION, ERA or COMPLIANCE (old vs new, legacy, deprecated,
    rebrand), then gradient-vs-flat, bevel-vs-flat, exact palette and finish are often THE
    deciding evidence. Make them explicit inclusion or exclusion criteria.
  - If the goal is to find a mark/component regardless of styling, colour must NOT disqualify:
    the same asset is routinely recoloured, inverted, monochrome or knocked out on dark.
  - Shape, proportion, typography and sub-element layout are ALWAYS criteria.
For PERSON_SEARCH: bone structure only.
For EXACT_MATCH: require the same composition and subject; allow rescaling, re-encoding,
compression artefacts and minor cropping.

STEP 4 — RETRIEVAL SIGNALS
intent_query: ONE dense sentence of visual nouns describing the target — it is embedded on its
own as a search vector, so imperatives like "find all" are wasted tokens.
search_keywords: 5-8 single words, OR-matched against image descriptions.
tag_guesses: 4-8 free-text Cloud Vision label guesses. They are fuzzy-matched against the real
corpus vocabulary afterwards, so approximate guesses are fine.

STEP 5 — ADJUDICATION PROTOCOL
{ADJUDICATION_TEMPLATE}
"""


def build_audit_config(goal: str, image_bytes: Optional[bytes] = None,
                       mime: str = "image/png") -> dict:
    parts: List[Any] = []
    if image_bytes:
        parts.append(types.Part.from_bytes(data=image_bytes, mime_type=mime))
    parts.append(_forensic_prompt(goal, image_bytes is not None))
    r = client.models.generate_content(
        model=ORCHESTRATOR_MODEL, contents=parts,
        config=types.GenerateContentConfig(response_mime_type="application/json",
                                           response_schema=AuditConfig, temperature=0.0))
    return json.loads(r.text)

## Stage 2 — Retrieval

Four arms, all index-served, all against the unique table.

**Dual vectors.** `vec_image` embeds the reference image *alone*; `vec_text` embeds
`intent_query` *alone*. Because Gemini Embedding 2 aggregates multiple inputs into one vector,
combining them in a single call would blend the two — so they are embedded separately and fused
by RRF instead. "This exact logo" and "the old generation of it" stay independent signals.

**The negative vector is folded into the query** (`q = normalize(pos − w·neg)`) rather than
subtracted in SQL, which keeps lookalike suppression while remaining an indexable ANN query.

In [ ]:
def _fold_negative(pos: List[float], neg: List[float], w: float) -> List[float]:
    p, n = np.asarray(pos, np.float32), np.asarray(neg, np.float32)
    q = p - w * n
    return (q / (np.linalg.norm(q) + 1e-9)).tolist()


def build_query_vectors(cfg: dict, image_bytes: Optional[bytes],
                        face_crop: Optional[bytes]) -> Dict[str, Optional[List[float]]]:
    """Independent image and text vectors — never blended into one call."""
    mode = cfg.get("search_mode", "GENERAL_ASSET")
    vecs: Dict[str, Optional[List[float]]] = {"image": None, "text": None, "face": None}

    if face_crop:   # PERSON_SEARCH: anchor on the face, not the whole reference frame
        vecs["face"] = _embed([types.Part.from_bytes(data=face_crop, mime_type="image/jpeg")], FACE_DIM)
    if image_bytes:
        src = face_crop or image_bytes
        vecs["image"] = _embed([types.Part.from_bytes(
            data=src, mime_type="image/jpeg" if face_crop else "image/png")])

    intent = cfg.get("intent_query") or cfg.get("audit_goal") or ""
    if cfg.get("brand_or_identity"):
        intent = f"{intent}. {cfg['brand_or_identity']}"
    if intent.strip():
        vecs["text"] = _embed([intent[:2000]])

    w = NEGATIVE_WEIGHT.get(mode, 0.0)
    if w > 0 and cfg.get("exclusion_criteria"):
        neg = _embed(["; ".join(cfg["exclusion_criteria"])[:2000]])
        for k in ("image", "text"):
            if vecs[k]:
                vecs[k] = _fold_negative(vecs[k], neg, w)
    return vecs


async def resolve_tags(guesses: List[str], limit: int = 25) -> List[str]:
    """Map free-text guesses onto REAL corpus tags via trigram similarity.
    Nothing from the 70k vocabulary ever enters a prompt, so no token blow-up, no 429."""
    if not guesses:
        return []
    try:
        async with db_conn() as db:
            rows = await db.fetch(f"""
                SELECT v.tag, max(similarity(v.tag, g)) s, v.freq
                FROM {DB_SCHEMA}.{VOCAB_VIEW} v, unnest($1::text[]) g
                WHERE v.tag % g GROUP BY v.tag, v.freq
                ORDER BY s DESC, v.freq DESC LIMIT $2""", guesses, limit)
            return [r["tag"] for r in rows]
    except Exception as e:
        print(f"   tag resolution unavailable ({e}) — continuing without tag boost")
        return []


SELECT_COLS = ("asset_id, content_hash, gcs_raw_path, format, vision_tags, "
               "gemini_description, asset_filename, page_url")


# You originally wanted vision_tags to "limit the scan scope". That was a workaround for the
# missing ANN index — scanning 99L rows was slow, so narrowing the set helped. With the unique
# table + HNSW the scan is no longer the bottleneck, and pre-filtering now COSTS recall: a tag
# filter that is even slightly wrong silently removes true matches the vector arm would have
# found, and a filtered ANN degrades toward a scan of the filtered set anyway.
# So it is OFF by default and tags act as a precision boost instead. Turn it on only when the
# resolved tags are unambiguous and the corpus slice is genuinely huge.
TAG_SCOPE_FILTER = False

async def _arm_vector(vec, mode, limit, scope_tags: Optional[List[str]] = None) -> List[dict]:
    """Pure ANN. This exact shape — ORDER BY <=> ... LIMIT — is what HNSW accelerates."""
    if not vec:
        return []
    use_scope = bool(TAG_SCOPE_FILTER and scope_tags)
    where = "WHERE vision_tags && $3::text[] " if use_scope else ""
    async with db_conn(vector=True) as db:
        async with db.transaction():
            await db.execute(f"SET LOCAL hnsw.ef_search = {EF_SEARCH.get(mode, 120)}")
            sql = (f"SELECT {SELECT_COLS}, embedding <=> $1::vector AS vector_distance "
                   f"FROM {TBL()} {where}ORDER BY embedding <=> $1::vector LIMIT $2")
            args = [vec, limit] + ([scope_tags] if use_scope else [])
            rows = await db.fetch(sql, *args)
    return [dict(r) for r in rows]


async def _arm_fts(keywords: List[str], limit: int) -> List[dict]:
    """OR-joined to_tsquery. v1 used plainto_tsquery, which ANDs every term — a recall killer."""
    terms = [re.sub(r"[^\w]", "", k) for k in keywords or []]
    q = " | ".join(t for t in terms if t)
    if not q:
        return []
    async with db_conn() as db:
        rows = await db.fetch(
            f"SELECT {SELECT_COLS}, ts_rank_cd(to_tsvector('english', coalesce(gemini_description,'')), "
            f"       to_tsquery('english', $1)) AS fts_rank "
            f"FROM {TBL()} "
            f"WHERE to_tsvector('english', coalesce(gemini_description,'')) @@ to_tsquery('english', $1) "
            f"ORDER BY fts_rank DESC LIMIT $2", q, limit)
    return [dict(r) for r in rows]


async def _arm_face(face_vec, limit: int) -> Tuple[List[dict], bool]:
    """ANN over per-face embeddings — how a 30px corner face becomes findable.
    Falls back to a tag net (weak, unranked) when asset_faces is absent."""
    has_faces = (globals().get("PREFLIGHT", {}) or {}).get(FACES_TABLE, False)
    if face_vec and has_faces:
        async with db_conn(vector=True) as db:
            async with db.transaction():
                await db.execute(f"SET LOCAL hnsw.ef_search = {EF_SEARCH['PERSON_SEARCH']}")
                rows = await db.fetch(f"""
                    WITH nf AS (SELECT content_hash, face_embedding <=> $1::vector AS d,
                                       face_area_frac
                                FROM {DB_SCHEMA}.{FACES_TABLE}
                                ORDER BY face_embedding <=> $1::vector LIMIT $2),
                         best AS (SELECT content_hash, min(d) d, max(face_area_frac) af
                                  FROM nf GROUP BY content_hash)
                    SELECT u.{SELECT_COLS.replace(', ', ', u.')}, b.d AS face_distance, b.af AS face_area_frac
                    FROM best b JOIN {TBL()} u USING (content_hash)
                    ORDER BY b.d LIMIT $2""", face_vec, limit)
        return [dict(r) for r in rows], True

    async with db_conn() as db:
        rows = await db.fetch(f"""
            SELECT {SELECT_COLS},
                   (SELECT count(*) FROM unnest(vision_tags) t WHERE t ILIKE ANY($1::text[])) AS tag_hits
            FROM {TBL()} WHERE vision_tags && $2::text[]
            ORDER BY tag_hits DESC LIMIT $3""",
            [f"%{t}%" for t in PERSON_FACE_TAGS], PERSON_FACE_TAGS, limit)
    return [dict(r) for r in rows], False


def rerank(items: List[dict], cfg: dict, tags: List[str]) -> List[dict]:
    """Bounded, additive. Max ~2.7× total (10× only for a confirmed near-exact hit).

    v1 multiplied by up to ~48× on noisy signals, so a rank-#3000 row with three generic tags
    beat the true #1 vector match by ~24×. Boosts must never outrank similarity itself —
    this stage only decides who reaches the LLM, so it favours recall and stability."""
    mode = cfg.get("search_mode", "GENERAL_ASSET")
    tagset = {t.lower() for t in tags}
    for it in items:
        d = it.get("vector_distance", 1.0)
        row_tags = {str(t).lower() for t in (it.get("vision_tags") or [])}
        bonus = 0.0
        if tagset & row_tags:
            bonus += 0.15                                   # tag agreement: a nudge, not a verdict
        bonus += 0.25 * math.exp(-3.0 * d)                  # smooth vector proximity, ≤0.25
        if mode == "PERSON_SEARCH" and it.get("face_distance") is not None:
            bonus += 0.40 * math.exp(-3.0 * float(it["face_distance"]))
        if mode == "EXACT_MATCH" and d < 0.05:
            bonus += 9.0                                    # a true near-duplicate; let it dominate
        it["relevance_score"] = it["base_rrf"] * (1.0 + bonus)
        it["rerank_boost"] = 1.0 + bonus
    items.sort(key=lambda x: x["relevance_score"], reverse=True)
    return items


async def hybrid_search(cfg: dict, image_bytes: Optional[bytes] = None,
                        face_crop: Optional[bytes] = None) -> pd.DataFrame:
    mode = cfg.get("search_mode", "GENERAL_ASSET")
    t0 = time.time()

    vecs = build_query_vectors(cfg, image_bytes, face_crop)
    tags = await resolve_tags(cfg.get("tag_guesses", []))
    if tags:
        print(f"   tags resolved → {tags[:8]}")

    face_vec = vecs["face"] if mode == "PERSON_SEARCH" else None
    r_img, r_txt, r_fts, (r_face, real_faces) = await asyncio.gather(
        _arm_vector(vecs["image"], mode, ARM_LIMIT, tags),
        _arm_vector(vecs["text"], mode, ARM_LIMIT, tags),
        _arm_fts(cfg.get("search_keywords", []), ARM_LIMIT),
        _arm_face(face_vec, ARM_LIMIT) if mode == "PERSON_SEARCH" else _noface(),
    )
    print(f"   arms: image={len(r_img)} text={len(r_txt)} fts={len(r_fts)} face={len(r_face)}"
          + ("" if not (mode == "PERSON_SEARCH") else
             f" [{'face-embeddings' if real_faces else 'TAG FALLBACK — build asset_faces for real recall'}]"))

    # ── RRF, renormalized over the arms that actually returned ──
    arms = {"vec_image": r_img, "vec_text": r_txt, "fts": r_fts, "face": r_face}
    w = dict(RRF_WEIGHTS.get(mode, RRF_WEIGHTS["GENERAL_ASSET"]))
    live = {k: v for k, v in w.items() if arms[k]}
    tot = sum(live.values())
    if tot <= 0:                       # every live arm has weight 0 for this mode
        live = {k: 1.0 / len(live) for k in live} if live else {}
    else:
        live = {k: v / tot for k, v in live.items()}

    fused: Dict[str, dict] = {}
    for arm, weight in live.items():
        for rank, row in enumerate(arms[arm]):
            key = row.get("content_hash") or row["gcs_raw_path"]
            cur = fused.setdefault(key, {**row, "base_rrf": 0.0, "arms": []})
            cur["base_rrf"] += weight / (RRF_K + rank + 1)
            cur["arms"].append(arm)
            for f in ("vector_distance", "face_distance"):
                if row.get(f) is not None and cur.get(f) is None:
                    cur[f] = row[f]

    items = rerank(list(fused.values()), cfg, tags)
    print(f"   {len(items)} unique candidates in {time.time()-t0:.2f}s")
    return pd.DataFrame(items)


async def _noface():
    return [], False


def select_candidates(df: pd.DataFrame, mode: str) -> pd.DataFrame:
    """Hard budget + a recall floor. Replaces Kneedle and its three unbounded rescue passes,
    which could push every face-tagged row into inference and make cost unpredictable."""
    if df.empty:
        return df
    n = BUDGET.get(mode, 40)
    top = df.head(n)
    if "vector_distance" in df.columns:                     # never drop a very close match
        rescue = df.iloc[n:]
        rescue = rescue[rescue["vector_distance"] < SAFEGUARD_DIST.get(mode, 0.35)]
        if not rescue.empty:
            print(f"   +{len(rescue)} rescued under distance {SAFEGUARD_DIST.get(mode,0.35)}")
            top = pd.concat([top, rescue], ignore_index=True)
    if len(df) > len(top):
        print(f"   NOTE: {len(df)-len(top)} candidates NOT audited (budget {n}). Coverage is partial.")
    return top.reset_index(drop=True)

## Exact match — answered without the LLM

`content_hash` equality is O(1) and exact. This turns the slowest, most expensive mode into
the fastest one, and directly answers *"find all the pages where this image appears"* by
expanding back over the full 99L occurrence table.

In [ ]:
import hashlib

# Which digest your ingest job used is not recorded anywhere in the schema, so try the
# common ones and match on whichever the corpus actually stores.
HASH_ALGOS = ("sha256", "md5", "sha1")

def content_hashes(b: bytes) -> List[str]:
    return [hashlib.new(a, b).hexdigest() for a in HASH_ALGOS]


async def exact_match(image_bytes: bytes) -> pd.DataFrame:
    hs = content_hashes(image_bytes)
    async with db_conn() as db:
        rows = await db.fetch(
            f"SELECT asset_id, gcs_raw_path, page_url, asset_filename, content_hash "
            f"FROM {DB_SCHEMA}.{BASE_TABLE} WHERE content_hash = ANY($1::text[])", hs)
    if rows:
        algo = HASH_ALGOS[hs.index(rows[0]["content_hash"])] if rows[0]["content_hash"] in hs else "?"
        print(f"   exact hit via {algo} → {len(rows)} occurrences")
    else:
        print("   no byte-identical row (or content_hash uses a digest not in HASH_ALGOS)")
    return pd.DataFrame([dict(r) for r in rows])


async def pages_for(hashes: List[str]) -> pd.DataFrame:
    """Expand deduped results back to every page each image appears on."""
    if not hashes:
        return pd.DataFrame()
    async with db_conn() as db:
        rows = await db.fetch(
            f"SELECT content_hash, page_url, gcs_raw_path FROM {DB_SCHEMA}.{BASE_TABLE} "
            f"WHERE content_hash = ANY($1::text[])", hashes)
    return pd.DataFrame([dict(r) for r in rows])

## Stage 3 — LLM adjudication

Every call sends **IMAGE 0 (reference) + IMAGE 1 (candidate) + the audit config**, and the
protocol forces an explicit side-by-side comparison rather than judging the candidate alone.

v1 stacked a hardcoded mode protocol *and* the generated `adjudication_logic` *and* the criteria
*and* general instructions — heavy overlap that diluted attention across four near-duplicate
instruction blocks. Here the generated protocol is the single source of steps, preceded by a
short constant persona describing **how a human reviewer looks at an image**.

IMAGE 0 and the persona are byte-identical across every candidate in a run, so they are
**context-cached** once instead of re-uploaded per call.

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from PIL import Image

AUDITOR_PERSONA = """\
You are a meticulous visual-asset auditor. You inspect an image the way a trained human reviewer
does, and you never take shortcuts:

- You scan the ENTIRE canvas before judging — corners, edges, and background included.
- You zoom into small regions. Targets are often 30px, embedded inside a screenshot, hero banner,
  device frame or app screen, cropped at an edge, rotated, low-contrast or partly occluded.
- A target that is NOT isolated on a clean background is still a match. Being embedded in a
  larger composite is normal and is never itself a reason to fail.
- You compare against the reference feature by feature, never on overall vibe or colour alone.
- The same mark is routinely recoloured, inverted or rendered monochrome. Colour alone never
  disqualifies unless the criteria explicitly demand a specific colour.
- You state what you actually see, including when you are unsure — an honest low confidence is
  more useful than a confident guess."""


def flatten_alpha(b: bytes, bg=(30, 30, 30)) -> bytes:
    """Composite transparency onto dark grey — white-on-transparent logos are otherwise invisible."""
    try:
        im = Image.open(io.BytesIO(b))
        if im.mode in ("RGBA", "LA") or (im.mode == "P" and "transparency" in im.info):
            im = im.convert("RGBA")
            out = Image.alpha_composite(Image.new("RGBA", im.size, bg + (255,)), im).convert("RGB")
            buf = io.BytesIO(); out.save(buf, format="PNG")
            return buf.getvalue()
    except Exception:
        pass
    return b


def build_protocol(cfg: dict, has_ref: bool) -> str:
    inc = "\n".join(f"  - {c}" for c in cfg.get("inclusion_criteria", [])) or "  - None"
    exc = "\n".join(f"  - {c}" for c in cfg.get("exclusion_criteria", [])) or "  - None"
    layout = ("IMAGE 0 = reference uploaded by the user (the target).\n"
              "IMAGE 1 = candidate under audit.\n"
              "IMAGE 0 may include device frames, hands or surrounding UI noise — compare ONLY the\n"
              "target element inside it, not its framing.\n" if has_ref else
              "IMAGE 0 = the candidate under audit. No reference image was provided.\n")
    return f"""{AUDITOR_PERSONA}

=== IMAGE LAYOUT ===
{layout}
=== AUDIT CONTEXT ===
Search mode : {cfg.get('search_mode')}   Breadth: {cfg.get('intent_breadth','SAME_BRAND_ANY_VARIANT')}
Goal        : {cfg.get('audit_goal')}
Reference   : {(cfg.get('image_description') or 'see IMAGE 0')[:600]}
Identity    : {cfg.get('brand_or_identity') or 'n/a'}

Inclusion — the asset must satisfy ALL:
{inc}

Exclusion — the asset FAILS if ANY trigger:
{exc}

=== ADJUDICATION PROTOCOL (execute every step, in order) ===
{cfg.get('adjudication_logic','')}

{cfg.get('audit_instructions','')}

Write your findings for EVERY step into visual_analysis_step_by_step before deciding.
If the target is genuinely absent, say so plainly — do not invent a partial match.
If you are unsure, lower match_confidence rather than guessing; a threshold handles the rest."""


def _verdict_model(cfg: dict):
    f: Dict[str, Any] = {
        "visual_analysis_step_by_step": (str, Field(description=(
            "MANDATORY. Execute every numbered step of the protocol and record findings per step. "
            "For a small or embedded target, state WHERE you found it. Never jump to a verdict."))),
        "target_located": (bool, Field(description="Was the target element found anywhere in the candidate?")),
        "target_location": (str, Field(description=(
            "Where it sits and roughly how large, e.g. 'top-right ~40px inside a phone mockup'. "
            "'not found' if absent."))),
        "matches_criteria": (bool, Field(description=(
            "Final verdict. True ONLY if ALL inclusion criteria hold and NO exclusion triggers."))),
        "match_confidence": (int, Field(description=(
            "0-100. Lower it for occlusion, small size, blur, odd angle or any ambiguity."))),
        "match_rationale": (str, Field(description=(
            "One paragraph citing the specific visual evidence behind the verdict."))),
    }
    for k, v in (cfg.get("extraction_schema") or {}).items():
        if k in f:
            continue
        t = str(v).lower()
        py = bool if "bool" in t else int if "int" in t else float if ("float" in t or "number" in t) else str
        f[k] = (py, Field(description=f"Extracted: {k}"))
    return create_model("Verdict", **f)


async def audit_one(row: dict, cfg: dict, model, ex, ref_part, cache_name, prompt) -> dict:
    loop = asyncio.get_running_loop()
    try:
        img = await loop.run_in_executor(ex, lambda: flatten_alpha(_bytes(row["gcs_raw_path"])))
        cand = types.Part.from_bytes(data=img, mime_type="image/png")

        if cache_name:                       # IMAGE 0 + persona already cached
            contents, gcfg = [cand], types.GenerateContentConfig(
                cached_content=cache_name, response_mime_type="application/json",
                response_schema=model, temperature=0.0)
        else:
            contents = ([ref_part, cand, prompt] if ref_part else [cand, prompt])
            gcfg = types.GenerateContentConfig(response_mime_type="application/json",
                                               response_schema=model, temperature=0.0)

        r = await loop.run_in_executor(ex, lambda: client.models.generate_content(
            model=INFERENCE_MODEL, contents=contents, config=gcfg))
        return {**row, **json.loads(r.text)}
    except Exception as e:
        return {**row, "matches_criteria": False, "match_confidence": 0, "target_located": False,
                "match_rationale": f"audit failed: {e}", "error": str(e)}


async def run_audit(df: pd.DataFrame, cfg: dict, ref_bytes: Optional[bytes]) -> pd.DataFrame:
    if df.empty:
        return df
    mode = cfg.get("search_mode", "GENERAL_ASSET")
    model, prompt = _verdict_model(cfg), build_protocol(cfg, ref_bytes is not None)
    ref_part = types.Part.from_bytes(data=flatten_alpha(ref_bytes), mime_type="image/png") if ref_bytes else None

    # Cache IMAGE 0 + the static protocol once: identical across every candidate in this run.
    cache_name = None
    if USE_CONTEXT_CACHE and ref_part:
        try:
            cache_name = client.caches.create(
                model=INFERENCE_MODEL,
                config=types.CreateCachedContentConfig(
                    contents=[types.Content(role="user", parts=[ref_part, types.Part.from_text(text=prompt)])],
                    ttl=CACHE_TTL)).name
            print(f"   context cache active — reference sent once, not {len(df)}×")
        except Exception as e:
            print(f"   context cache unavailable ({str(e)[:90]}) — inlining per call")

    print(f"   auditing {len(df)} candidates · mode={mode} · workers={MAX_WORKERS}")
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
        rows = await asyncio.gather(*[
            audit_one(r, cfg, model, ex, ref_part, cache_name, prompt)
            for r in df.to_dict("records")])
    dt = time.time() - t0
    print(f"   done in {dt:.1f}s ({len(rows)/max(dt,0.01):.1f} img/s)")

    if cache_name:
        try:
            client.caches.delete(name=cache_name)
        except Exception:
            pass

    out = pd.DataFrame(rows)
    # Confidence guardrail: demote weak PASSes.
    thr = CONF_THRESHOLD.get(mode, 70) + BREADTH_CONF_ADJ.get(cfg.get("intent_breadth"), 0)
    weak = out["matches_criteria"].fillna(False).astype(bool) & (out["match_confidence"].fillna(0) < thr)
    out.loc[weak, "match_rationale"] = f"[below {thr}% confidence] " + out.loc[weak, "match_rationale"].astype(str)
    out.loc[weak, "matches_criteria"] = False
    if weak.any():
        print(f"   guardrail demoted {int(weak.sum())} low-confidence PASS → FAIL")
    return out.sort_values("relevance_score", ascending=False).reset_index(drop=True)

## End-to-end

In [ ]:
def _mime(p: str) -> str:
    p = p.lower()
    return ("image/png" if p.endswith(".png") else "image/webp" if p.endswith(".webp")
            else "image/gif" if p.endswith(".gif") else "image/jpeg")


async def stage1(goal: str, ref_path: Optional[str] = None) -> dict:
    """Forensics + audit config in ONE Pro call, plus a single face detection for person mode."""
    img = _bytes(ref_path) if ref_path else None
    print("1. Building audit config…")
    cfg = build_audit_config(goal, img, _mime(ref_path) if ref_path else "image/png")
    print(json.dumps({k: v for k, v in cfg.items() if k != "extraction_schema"}, indent=2)[:2200])

    cfg["_ref_bytes"], cfg["_face_crop"] = img, None
    if cfg.get("search_mode") == "PERSON_SEARCH" and img:
        faces = detect_faces(img)
        if faces:
            big = max(faces, key=lambda f: f["area_frac"])
            cfg["_face_crop"] = crop_face(img, big["bbox"])
            print(f"   face detected ({big['area_frac']*100:.1f}% of frame) — embedding the crop, "
                  f"not the whole reference")
        else:
            print("   no face detected — falling back to the full reference image")
    return cfg


async def stage2(cfg: dict, expand_pages: bool = True) -> pd.DataFrame:
    mode, ref = cfg.get("search_mode", "GENERAL_ASSET"), cfg.get("_ref_bytes")
    t0 = time.time()

    if mode == "EXACT_MATCH" and ref:
        exact = await exact_match(ref)
        if not exact.empty:
            print(f"\nExact byte-identical matches found — no LLM inference needed "
                  f"({time.time()-t0:.2f}s total).")
            return exact
        print("   no byte-identical copy; falling through to near-duplicate vector search")

    print(f"\n2. Retrieval (mode={mode})…")
    df = await hybrid_search(cfg, ref, cfg.get("_face_crop"))
    if df.empty:
        print("   no candidates."); return df

    print("\n3. Selection…")
    df = select_candidates(df, mode)

    print("\n4. Adjudication…")
    res = await run_audit(df, cfg, ref)

    if expand_pages and "content_hash" in res.columns:
        hits = res[res["matches_criteria"] == True]["content_hash"].dropna().tolist()
        if hits:
            pages = await pages_for(hits)
            print(f"\n   {len(hits)} matching images appear on {pages['page_url'].nunique()} pages")
            globals()["matched_pages"] = pages

    passed = int(res["matches_criteria"].sum()) if not res.empty else 0
    print(f"\n{'='*64}\nPASS {passed}/{len(res)} · total {time.time()-t0:.2f}s\n{'='*64}")
    for i, r in res.head(40).iterrows():
        name = (r.get("asset_filename") or str(r.get("gcs_raw_path", ""))).split("/")[-1]
        print(f"  [{i+1:>3}] {'PASS' if r.get('matches_criteria') else 'FAIL'} "
              f"conf={r.get('match_confidence',0):>3} d={r.get('vector_distance',1):.3f} "
              f"{name[:44]:<44} {str(r.get('target_location',''))[:34]}")
    return res


def show(df: pd.DataFrame, only_pass: bool = False, n: int = 30):
    """Inline thumbnails + verdicts."""
    import base64
    from IPython.display import display, HTML
    d = df[df["matches_criteria"] == True] if only_pass else df
    if d.empty:
        print("nothing to show"); return
    html = ["<table style='font:12px sans-serif'><tr><th>img</th><th>verdict</th>"
            "<th>conf</th><th>dist</th><th>where</th><th>why</th><th>page</th></tr>"]
    for _, r in d.head(n).iterrows():
        try:
            b64 = base64.b64encode(_bytes(r["gcs_raw_path"])).decode()
            im = f"<img src='data:image/png;base64,{b64}' style='max-width:130px;max-height:130px'>"
        except Exception:
            im = "—"
        ok = bool(r.get("matches_criteria"))
        html.append(
            f"<tr><td>{im}</td>"
            f"<td style='color:{'#0a0' if ok else '#a00'}'><b>{'PASS' if ok else 'FAIL'}</b></td>"
            f"<td>{r.get('match_confidence','')}</td><td>{r.get('vector_distance',float('nan')):.3f}</td>"
            f"<td>{str(r.get('target_location',''))[:44]}</td>"
            f"<td style='max-width:420px'>{str(r.get('match_rationale',''))[:340]}</td>"
            f"<td><a href='{r.get('page_url','')}' target='_blank'>link</a></td></tr>")
    display(HTML("".join(html) + "</table>"))

In [ ]:
# ── TEST RUN · Stage 1 ────────────────────────────────────────────────────────
reference_image_path = None
try:
    from google.colab import files
    print("Optional: upload a reference image (skip for text-only search)")
    up = files.upload()
    if up:
        reference_image_path = list(up.keys())[0]
except Exception:
    pass

GOAL = "Find all pages using the old version of this logo"

audit_cfg = await stage1(GOAL, reference_image_path)

In [ ]:
# ── TEST RUN · Stage 2 ────────────────────────────────────────────────────────
# Override anything the model got wrong before spending inference budget, e.g.:
# audit_cfg["search_mode"] = "LOGO_SIMILARITY"
# audit_cfg["exclusion_criteria"].append("Exclude the current flat redesign introduced in 2020")

results = await stage2(audit_cfg)
show(results, only_pass=False)